# Rapport final — Classification de trajectoires par LSTM

Notebook de synthèse: il décrit la démarche et les résultats, en alternant **explications (Markdown)** et **extraits de code**.
Le code complet (entraînement, visualisations, etc.) est disponible dans `classification.ipynb` (fourni avec le rendu).


## Objectif

Le but du projet est de **classer des trajectoires** d’un bras robotique (tâche *peg-in-hole*) en **3 classes**, à partir de séquences temporelles.
On entraîne un **LSTM** (réseau récurrent) qui prend une trajectoire ` (time, features) ` en entrée et prédit la classe.


## 1) Visualisation des données (avant modélisation)

Avant d’entraîner un modèle, on commence par visualiser les trajectoires pour:
- vérifier que les données sont cohérentes (format, amplitudes, dynamique)
- observer si les classes semblent séparables sur les composantes de position

<table>
<tr>
<td><img src="images/3D_viz_data.png" width="450"/></td>
<td><img src="images/2D_viz_data.png" width="450"/></td>
</tr>
</table>

Ces figures montrent que la dynamique (forme globale, évolution temporelle) varie selon la classe, ce qui motive l’usage d’un modèle séquentiel (LSTM).

Remarque: des versions interactives Plotly sont sauvegardées dans `trajectories_3d.html` et `trajectories_2d.html`.


In [ ]:
# (extrait) Exemple de projection sur (x, y, z)
traj = data[traj_idx]  # shape: (150, 20)
x, y, z = traj[:, 0], traj[:, 1], traj[:, 2]


## 2) Extraction et préparation des données

### 2.1 Format
Les fichiers fournis sont des tenseurs NumPy de forme `(n_samples, n_timesteps, n_features)`.
Dans notre cas:
- `n_timesteps = 150`
- `n_features = 20`
- 3 classes, avec `256` trajectoires par classe (jeu équilibré)

Les 20 features correspondent à:
- `peg_pose` (7): position + quaternion
- `peg_vel`  (6): vitesse linéaire + angulaire
- `hole_pos` (7): position + quaternion

### 2.2 Chargement + labels
On concatène les 3 fichiers, puis on construit les labels (0/1/2).


In [ ]:
import numpy as np

data_class1 = np.load('class1_trajectories.npy')
data_class2 = np.load('class2_trajectories.npy')
data_class3 = np.load('class3_trajectories.npy')

data = np.concatenate([data_class1, data_class2, data_class3], axis=0)
labels = np.concatenate([
    np.zeros(len(data_class1)),
    np.ones(len(data_class2)),
    2 * np.ones(len(data_class3)),
])

print('data:', data.shape, 'labels:', labels.shape)


### 2.3 Split train/val/test + reproductibilité
Pour comparer correctement les runs, on fixe un seed.
On split ensuite en **80% / 10% / 10%**:
- train: 614
- val: 76
- test: 78


In [ ]:
import torch

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

n_samples = len(data)
indices = np.arange(n_samples)
np.random.shuffle(indices)

train_size = int(0.8 * n_samples)
val_size = int(0.1 * n_samples)
test_size = n_samples - train_size - val_size

train_idx = indices[:train_size]
val_idx = indices[train_size:train_size+val_size]
test_idx = indices[train_size+val_size:]

train_data, train_labels = data[train_idx], labels[train_idx]
val_data, val_labels = data[val_idx], labels[val_idx]
test_data, test_labels = data[test_idx], labels[test_idx]

print(train_data.shape, val_data.shape, test_data.shape)


### 2.4 Normalisation (z-score)
Dans les runs finaux, on applique une normalisation **z-score** calculée sur le train (puis appliquée à val/test).
On protège `std` pour éviter la division par 0.


In [ ]:
# (extrait) z-score à partir du train
train_flat = train_data.reshape(-1, train_data.shape[-1])  # (n_train*150, 20)
mean = train_flat.mean(axis=0)
std = train_flat.std(axis=0)
std = np.where(std == 0, 1e-6, std)

train_norm = (train_data - mean) / std
val_norm   = (val_data   - mean) / std
test_norm  = (test_data  - mean) / std


## 3) Modèle: LSTM pour la classification

On utilise un LSTM car la classe dépend de **l’évolution temporelle** des features.
Architecture finale (celle qui donne les meilleurs résultats):
- `LSTM`: 2 couches (`num_layers=2`), 64 unités cachées (`hidden_size=64`)
- `Dropout=0.2`
- `Linear(64 → 3)`
- `CrossEntropyLoss`, `Adam(lr=1e-3)`
- `batch_size=32`, `epochs=40`


In [ ]:
import torch.nn as nn

class MonModele(nn.Module):
    def __init__(self, input_size=20, hidden_size=64, num_layers=2, output_size=3, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=dropout)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        x, _ = self.lstm(x)
        x = x[:, -1, :]      # dernier pas de temps
        x = self.dropout(x)
        return self.fc(x)


## 4) Plan d’expérience (runs / hyperparamètres)

On a procédé par itérations, en modifiant progressivement:
1. la **normalisation** (avec / sans z-score)
2. la capacité du modèle (`hidden_size`)
3. la régularisation (**dropout**)
4. la stabilité de l’entraînement (**exploding gradients** → gradient clipping)
5. la reproductibilité (**seed**)
6. le budget d’entraînement (10 / 20 / 40 epochs)


## 5) Problèmes rencontrés

### 5.1 Exploding gradients
Les LSTM/RNN peuvent souffrir d’**explosion du gradient** (gradients très grands → entraînement instable).
On a corrigé ce problème avec le **gradient clipping**.

![illustration_gradient_exploding](images/gradient_explosion.png)


In [ ]:
# (extrait) Gradient clipping
loss.backward()
torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
optimizer.step()


## 6) Résultats et analyse

On résume les runs principaux (courbes `loss/accuracy`).
Les logs détaillés et le code complet sont dans `classification.ipynb` et les notes de runs dans `rapport.ipynb`.


### Run 1 — Architecture simple (hidden=100) + z-score, 10 epochs (résultat médiocre)
![model_1_loss_accuracy](images/model_1_loss_accuracy.png)

Observation: performances de validation limitées → nécessité d’explorer d’autres réglages.


### Run 2 — hidden=64 sans normalisation (gain important)
![model_2_loss_accuracy](images/model_2_loss_accuracy.png)

Résultat (test): **92.31%** (72/78).


### Run 3 — Ajout du dropout (régularisation)
![model_3_loss_accuracy](images/model_3_loss_accuracy.png)

Résultat (test): **98.72%** (77/78).


### Run 4 — Dropout + gradient clipping (stabilité) — meilleur run
![model_4_loss_accuracy](images/model_4_loss_accuracy.png)

Résultat (test): **100%** (dans `classification.ipynb`: `Test loss ≈ 0.0007`, `Test accu = 100.00%`).


### Analyse
- Les visualisations initiales confirment que les classes sont plausiblement séparables.
- Le **seed** est indispensable pour comparer les essais.
- Le **dropout** améliore la généralisation.
- Le **gradient clipping** stabilise l’apprentissage et résout le problème d’exploding gradients.


## 7) Conclusion

Nous avons entraîné un classifieur basé sur un **LSTM** qui atteint **100% de réussite sur le test interne** après stabilisation (gradient clipping) et régularisation (dropout).

Pistes d’amélioration possibles:
- étude de l’influence du **nombre de features** (par ex. uniquement position/vitesse)
- data augmentation légère (bruit faible, jitter temporel) sans dégrader la physique
- évaluation sur un **jeu de test externe** (enseignant) avec la fonction d’évaluation du notebook
